In [1]:
import numpy as np, torch, torch.nn as nn, torch.optim as optim, torchvision, os
from torch.optim import lr_scheduler
from torch.autograd import Variable
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
from google.colab import _message

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# I. Preparing dataset

## 1. Interact with Google Colab

In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Section 19 - Transfer Learning in Image Classification (Pytorch)"

Mounted at /content/drive
/content/drive/MyDrive/Section 19 - Transfer Learning in Image Classification (Pytorch)


In [4]:
print("📂 Current directory:", os.getcwd())
!apt install tree -y
!tree -L 2 -a --dirsfirst

📂 Current directory: /content/drive/MyDrive/Section 19 - Transfer Learning in Image Classification (Pytorch)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 38 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (160 kB/s)
Selecting previously unselected package tree.
(Reading database ... 126718 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...
.
├── hymenoptera_data
│   ├── train
│   └── val
└── main.ipynb

3 directories, 1 file


## 2. Build Data Loader

In [5]:
data_transforms = {
    'train': transforms.Compose([transforms.RandomResizedCrop(224),
                                 transforms.RandomHorizontalFlip(),
                                 transforms.ToTensor(),
                                 transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    ]),
    'val': transforms.Compose([transforms.Resize(256),
                               transforms.CenterCrop(224),
                               transforms.ToTensor(),
                               transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [6]:
data_dir = './hymenoptera_data'
image_datasets = {}

# Create dataset train
image_datasets['train'] = datasets.ImageFolder(os.path.join(data_dir, 'train'),data_transforms['train'])
class_to_idx_train = image_datasets['train'].class_to_idx

# Create dataset val
image_datasets['val'] = datasets.ImageFolder(os.path.join(data_dir, 'val'),data_transforms['val'])
image_datasets['val'].class_to_idx = class_to_idx_train

# Create dataloader
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4, shuffle=True) for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}

class_names_to_idx = {x: image_datasets[x].class_to_idx for x in ['train', 'val']}
for phase in ['train', 'val']:
    print(f"📂 {phase} class_to_idx:")
    print(class_names_to_idx[phase])

📂 train class_to_idx:
{'ants': 0, 'bees': 1}
📂 val class_to_idx:
{'ants': 0, 'bees': 1}


# II. Training model

## 1. Modifying the network

In [7]:
from torchvision.models import resnet18, ResNet18_Weights
weights = ResNet18_Weights.DEFAULT
model_conv = resnet18(weights=weights)

# Freeze all layers in the network
for params in model_conv.parameters(): params.requires_grad = False

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 135MB/s]


In [8]:
num_ftrs = model_conv.fc.in_features
model_conv.fc = nn.Linear(num_ftrs, 2)

In [9]:
model = model_conv.to(device)

In [10]:
crieton = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_conv.parameters(), lr = 0.001, momentum = 0.9)

In [11]:
def lr_scheduler(optimizer, epoch, init_lr=0.001, lr_decay_epoch=7):
    lr = init_lr * (0.1**(epoch // lr_decay_epoch))

    if epoch % lr_decay_epoch == 0: print("LR is set to {}".format(lr))
    for param_group in optimizer.param_groups: param_group['lr'] = lr

    return optimizer

## 2. Fine-tune the network

In [12]:
num_epcochs = 3

for epoch in range(num_epcochs):
    print(f"Epoch {epoch}/{num_epcochs - 1}")

    optimizer = lr_scheduler(optimizer, epoch, init_lr=0.001, lr_decay_epoch=7)

    correct = 0
    for images, labels in dataloaders['train']:
        images = images.to(device)
        labels = labels.to(device)

        # Backpropagation
        optimizer.zero_grad()
        outputs = model_conv(images)
        loss = crieton(outputs, labels)
        loss.backward()
        optimizer.step()

        # Predict on train dataset
        _, predicted = torch.max(outputs, dim = 1)
        correct += (predicted == labels).sum().item()

    train_acc = 100*correct/dataset_sizes['train']
    print("Epoch [{}/{}], loss: {:.4f}, train accuracy: {}%"
            .format(epoch + 1, num_epcochs, loss.item(), train_acc))

Epoch 0/2
LR is set to 0.001
Epoch [1/3], loss: 0.6490, train accuracy: 66.39344262295081%
Epoch 1/2
Epoch [2/3], loss: 0.4999, train accuracy: 76.22950819672131%
Epoch 2/2
Epoch [3/3], loss: 0.5035, train accuracy: 80.32786885245902%


# III. Testing and visualizing the result

In [13]:
model_conv.eval()

with torch.no_grad():
    correct = 0
    total = 0

    for images, labels in dataloaders['val']:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model_conv(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print("Test accuracy: {:.3f} %".format(100*correct/total))

Test accuracy: 95.425 %
